# LITE validation ablation — long-epoch run (Kaggle GPU)

The 100-epoch runs were under-trained: original LITE's best epoch was always 100, and PigAirwayPressure stayed at chance. This notebook reruns the ablation with a paper-like budget.

**Before running (right-hand panel → Settings):**
- **Accelerator:** GPU T4 ×2 (or P100)
- **Internet:** On (needs a phone-verified Kaggle account)
- **Code:** upload `lite_code.zip` as a **private** Kaggle dataset and attach it (Add Input → Your Datasets). No GitHub access needed.

**How to run:** click **Save Version → Save & Run All (Commit)**. It then runs in the background (up to ~12 h) and you can close the tab. Results appear in the version's **Output** tab.

- Results are written to `/kaggle/working/` after every run.
- If a run hits the time limit, add the previous version's output as input (Add Input → your notebook) and set `RESUME_FROM` in the settings cell; finished runs are skipped.
- To stay under the limit you can do one dataset per version with `RUN_SEMG` / `RUN_PIG`.
- PigAirwayPressure gets only `original` and `first_difference` (52 classes × 2 train samples, so a validation split is impossible).

In [1]:
!nvidia-smi -L || echo "No GPU! Settings -> Accelerator -> GPU"

GPU 0: Tesla T4 (UUID: GPU-b95dfa1f-daf3-8927-0b9d-f8751271e8d8)
GPU 1: Tesla T4 (UUID: GPU-94393451-1a9c-8a14-13ef-e07e0ab5cf1d)


## 2. Get the code and install dependencies

In [2]:
# Code comes from the private Kaggle dataset made from lite_code.zip (attached via Add Input).
import glob, os, shutil
hits = glob.glob("/kaggle/input/**/src/classifiers/lite.py", recursive=True)
assert hits, "Attach the lite_code dataset: Add Input -> Your Datasets"
code_dir = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
shutil.copytree(code_dir, "/tmp/lite_code", dirs_exist_ok=True)
print("Code copied from", code_dir)
%cd /tmp/lite_code
!pip install -q aeon codecarbon
import tensorflow as tf, sklearn, numpy, aeon
print("TF", tf.__version__, "| sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| aeon", aeon.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

Code copied from /kaggle/input/datasets/paramikris/lite-reconstruct
/tmp/lite_code
TF 2.21.0 | sklearn 1.6.1 | numpy 2.1.3 | aeon 1.6.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 3. Write the updated `run_ablation.py`

Same as the repo version plus: the `validation_refit` condition, a `--lr-patience` option, append-and-resume CSV writing, and per-run start logging.

In [3]:
%%writefile run_ablation.py
"""Run a paired LITE ablation without using the test set for model selection."""

import argparse
import csv
import time
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

from src.classifiers import LITE
from src.utils import load_data


class BestEpoch(tf.keras.callbacks.Callback):
    def __init__(self, monitor):
        super().__init__()
        self.monitor = monitor
        self.best_value = float("inf")
        self.best_epoch = 0
        self.epochs_seen = 0

    def on_epoch_end(self, epoch, logs=None):
        self.epochs_seen = epoch + 1
        value = (logs or {}).get(self.monitor)
        if value is not None and value < self.best_value:
            self.best_value = float(value)
            self.best_epoch = epoch + 1


def add_first_difference(x):
    differences = np.diff(x, axis=1, prepend=x[:, :1, :])
    return np.concatenate((x, differences), axis=2)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--datasets", nargs="+", default=["PigAirwayPressure", "SemgHandMovementCh2"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[0, 1, 2, 3, 4])
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--patience", type=int, default=10)
    parser.add_argument("--val-split", type=float, default=0.2)
    parser.add_argument("--output-dir", type=Path, default=Path("results_ablation"))
    parser.add_argument(
        "--conditions",
        nargs="+",
        choices=["original", "validation", "first_difference", "validation_and_difference", "validation_refit"],
        default=["original", "validation"],
    )
    parser.add_argument("--lr-patience", type=int, default=8,
                        help="ReduceLROnPlateau patience (LITE paper uses 50)")
    parser.add_argument("--verbose", type=int, default=0)
    args = parser.parse_args()

    if not 0 < args.val_split < 1:
        parser.error("--val-split must be between 0 and 1")
    if args.epochs < 1 or args.patience < 1:
        parser.error("--epochs and --patience must be positive")

    args.output_dir.mkdir(parents=True, exist_ok=True)
    results_path = args.output_dir / "results_per_run.csv"
    fields = ["dataset", "condition", "seed", "accuracy", "training_seconds", "epochs_run", "best_epoch"]
    # validation_refit: pick the epoch count with early stopping on a validation split,
    # then retrain from scratch on the full training set for that many epochs.
    all_conditions = [
        ("original", False, False),
        ("validation", True, False),
        ("first_difference", False, True),
        ("validation_and_difference", True, True),
        ("validation_refit", True, False),
    ]

    # Append so rerunning into an existing output dir never wipes earlier results.
    # Runs already in the CSV are skipped, so an interrupted job can simply be restarted.
    new_file = not results_path.exists()
    done = set()
    if not new_file:
        with results_path.open(newline="") as existing:
            done = {(r["dataset"], r["condition"], int(r["seed"])) for r in csv.DictReader(existing)}
    with results_path.open("a", newline="") as results_file:
        writer = csv.DictWriter(results_file, fieldnames=fields)
        if new_file:
            writer.writeheader()
        results_file.flush()

        for dataset in args.datasets:
            x_train, y_train, x_test, y_test = load_data(file_name=dataset)
            for seed in args.seeds:
                for condition, use_validation, use_difference in all_conditions:
                    if condition not in args.conditions:
                        continue
                    if (dataset, condition, seed) in done:
                        print(f"{dataset} | {condition} | seed={seed} | already done, skipping", flush=True)
                        continue
                    print(f"{time.strftime('%H:%M:%S')} start {dataset} | {condition} | seed={seed}", flush=True)
                    tf.keras.utils.set_random_seed(seed)
                    if use_validation:
                        x_fit, x_val, y_fit, y_val = train_test_split(
                            x_train, y_train, test_size=args.val_split,
                            random_state=seed, stratify=y_train,
                        )
                    else:
                        x_fit, y_fit = x_train, y_train
                        x_val, y_val = None, None

                    if use_difference:
                        x_fit = add_first_difference(x_fit)
                        x_test_run = add_first_difference(x_test)
                        if x_val is not None:
                            x_val = add_first_difference(x_val)
                    else:
                        x_test_run = x_test

                    run_dir = args.output_dir / dataset / condition / f"seed_{seed}"
                    run_dir.mkdir(parents=True, exist_ok=True)
                    monitor = "val_loss" if use_validation else "loss"
                    epoch_tracker = BestEpoch(monitor)
                    model = LITE(
                        output_directory=f"{run_dir}/", length_TS=x_fit.shape[1],
                        n_channels=x_fit.shape[2], n_classes=len(np.unique(y_train)),
                        n_epochs=args.epochs, verbose=args.verbose,
                    )
                    callbacks = [
                        tf.keras.callbacks.ReduceLROnPlateau(
                            monitor=monitor, factor=0.5, patience=args.lr_patience, min_lr=1e-4
                        ),
                        tf.keras.callbacks.ModelCheckpoint(
                            filepath=str(run_dir / "best_model.keras"),
                            monitor=monitor, save_best_only=True,
                        ),
                        epoch_tracker,
                    ]
                    if use_validation:
                        callbacks.append(tf.keras.callbacks.EarlyStopping(
                            monitor="val_loss", patience=args.patience,
                            restore_best_weights=True,
                        ))
                    model.callbacks = callbacks

                    started = time.perf_counter()
                    model.fit(
                        xtrain=x_fit, ytrain=y_fit, xval=x_val, yval=y_val,
                        plot_test=use_validation,
                    )
                    epochs_run = epoch_tracker.epochs_seen
                    best_epoch = epoch_tracker.best_epoch

                    if condition == "validation_refit":
                        tf.keras.utils.set_random_seed(seed)
                        refit_dir = run_dir / "refit"
                        refit_dir.mkdir(parents=True, exist_ok=True)
                        model = LITE(
                            output_directory=f"{refit_dir}/", length_TS=x_train.shape[1],
                            n_channels=x_train.shape[2], n_classes=len(np.unique(y_train)),
                            n_epochs=best_epoch, verbose=args.verbose,
                        )
                        model.callbacks = [
                            tf.keras.callbacks.ReduceLROnPlateau(
                                monitor="loss", factor=0.5, patience=args.lr_patience, min_lr=1e-4
                            ),
                            tf.keras.callbacks.ModelCheckpoint(
                                filepath=str(refit_dir / "best_model.keras"),
                                monitor="loss", save_best_only=True,
                            ),
                        ]
                        model.fit(xtrain=x_train, ytrain=y_train, plot_test=False)
                        epochs_run += best_epoch

                    training_seconds = time.perf_counter() - started
                    _, accuracy, _ = model.predict(xtest=x_test_run, ytest=y_test)
                    writer.writerow({
                        "dataset": dataset, "condition": condition, "seed": seed,
                        "accuracy": accuracy, "training_seconds": training_seconds,
                        "epochs_run": epochs_run,
                        "best_epoch": best_epoch,
                    })
                    results_file.flush()
                    print(f"{dataset} | {condition} | seed={seed} | accuracy={accuracy:.4f} | epochs={epoch_tracker.best_epoch}")


if __name__ == "__main__":
    main()

Overwriting run_ablation.py


## 4. Settings

`EPOCHS=1500` and `LR_PATIENCE=50` match the LITE paper. `PATIENCE` is the early-stopping patience for the validation conditions.

In [4]:
EPOCHS = 1500
PATIENCE = 100
LR_PATIENCE = 50
#SEEDS = "0 1 2 3 4"
SEEDS = "3 4"
OUT = f"/kaggle/working/ablation_{EPOCHS}ep"

RUN_SEMG = True   # SemgHand is running on the laptop GPU
RUN_PIG = False
SEMG_CONDITIONS = "original validation validation_refit first_difference validation_and_difference"
PIG_CONDITIONS = "original first_difference"

# To continue an interrupted version: Add Input -> this notebook's previous version, then e.g.
# RESUME_FROM = "/kaggle/input/<notebook-name>/ablation_1500ep"
RESUME_FROM = ""

import os, shutil
if RESUME_FROM and os.path.isdir(RESUME_FROM) and not os.path.exists(OUT):
    shutil.copytree(RESUME_FROM, OUT)
    print("Resumed previous results from", RESUME_FROM)
os.makedirs(OUT, exist_ok=True); print("Saving to", OUT)

Saving to /kaggle/working/ablation_1500ep


## 5. Quick smoke test (~1 min)

Checks that everything works before the long run. It writes to a throwaway folder.

In [5]:
!python -u run_ablation.py --datasets SemgHandMovementCh2 --seeds 0 --epochs 2 --patience 1 \
    --conditions original validation_refit --output-dir /tmp/smoke
!cat /tmp/smoke/results_per_run.csv && rm -rf /tmp/smoke

18:30:02 start SemgHandMovementCh2 | original | seed=0
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 93ms/step
SemgHandMovementCh2 | original | seed=0 | accuracy=0.1667 | epochs=2
18:30:37 start SemgHandMovementCh2 | validation_refit | seed=0
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step
SemgHandMovementCh2 | validation_refit | seed=0 | accuracy=0.1667 | epochs=2





## 6. SemgHandMovementCh2 — all 5 conditions

One line is printed when each run starts and when it finishes. To watch per-epoch progress, change `--verbose 0` to `--verbose 2` (very long output).

In [6]:
if RUN_SEMG:
    !python -u run_ablation.py --datasets SemgHandMovementCh2 --seeds {SEEDS} \
        --epochs {EPOCHS} --patience {PATIENCE} --lr-patience {LR_PATIENCE} \
        --conditions {SEMG_CONDITIONS} --output-dir {OUT}/semg --verbose 0

18:31:25 start SemgHandMovementCh2 | original | seed=3
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 95ms/step
SemgHandMovementCh2 | original | seed=3 | accuracy=0.5111 | epochs=1500
18:55:29 start SemgHandMovementCh2 | validation | seed=3
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step
SemgHandMovementCh2 | validation | seed=3 | accuracy=0.5156 | epochs=186
18:59:37 start SemgHandMovementCh2 | first_difference | seed=3
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step
SemgHandMovementCh2 | first_difference | seed=3 | accuracy=0.4244 | epochs=826
19:22:27 start SemgHandMovementCh2 | validation_and_difference | seed=3
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step
SemgHandMovementCh2 | validation_and_difference | seed=3 | accuracy=0.4311 | epochs=217
19:26:56 start SemgHandMovementCh2 | validation_refit | seed=3
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step
SemgHandMovementCh2 | validation_refit | seed=3 | accuracy=0.4978 | epochs=475
19:42:15 start SemgHandMovementCh2 | original | seed=4
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step
Semg

## 7. PigAirwayPressure — original and first_difference

In [7]:
if RUN_PIG:
    !python -u run_ablation.py --datasets PigAirwayPressure --seeds {SEEDS} \
        --epochs {EPOCHS} --patience {PATIENCE} --lr-patience {LR_PATIENCE} \
        --conditions {PIG_CONDITIONS} --output-dir {OUT}/pig --verbose 0

## 8. Results — paired comparison against original

In [8]:
import pandas as pd
from scipy import stats

frames = [pd.read_csv(f"{OUT}/{d}/results_per_run.csv") for d in ("semg", "pig") if os.path.exists(f"{OUT}/{d}/results_per_run.csv")]
df = pd.concat(frames, ignore_index=True)
df.to_csv(f"{OUT}/all_results.csv", index=False)

for dataset, g in df.groupby("dataset"):
    print(f"\n===== {dataset} =====")
    pivot = g.pivot(index="seed", columns="condition", values="accuracy")
    display(pivot.round(4))
    summary = g.groupby("condition").agg(
        mean_acc=("accuracy", "mean"), sd=("accuracy", "std"),
        mean_epochs=("epochs_run", "mean"), mean_best_epoch=("best_epoch", "mean"),
        mean_train_s=("training_seconds", "mean"), n=("seed", "count"))
    rows = []
    for c in pivot.columns:
        if c == "original" or "original" not in pivot: continue
        paired = pivot[["original", c]].dropna()
        d = paired[c] - paired["original"]
        p = stats.ttest_rel(paired[c], paired["original"]).pvalue if len(paired) > 1 else float("nan")
        rows.append({"condition": c, "diff_vs_original": d.mean(), "wins": f"{(d > 0).sum()}/{len(d)}", "paired_t_p": p})
    if rows:
        summary = summary.join(pd.DataFrame(rows).set_index("condition"))
    display(summary.round(4))


===== SemgHandMovementCh2 =====


condition,first_difference,original,validation,validation_and_difference,validation_refit
seed,,,,,
3,0.4244,0.5111,0.5156,0.4311,0.4978
4,0.4244,0.4756,0.4844,0.4289,0.4089


,mean_acc,sd,mean_epochs,mean_best_epoch,mean_train_s,n,diff_vs_original,wins,paired_t_p
condition,,,,,,,,,
first_difference,0.4244,0.0000,1500.0,948.5,1388.6496,2,-0.0689,0/2,0.1608
original,0.4933,0.0251,1500.0,1500.0,1417.4638,2,NaN,NaN,NaN
validation,0.5000,0.0220,303.0,203.0,254.1606,2,0.0067,2/2,0.2048
validation_and_difference,0.4300,0.0016,282.0,182.0,237.2240,2,-0.0633,0/2,0.1638
validation_refit,0.4533,0.0629,768.0,334.0,676.1398,2,-0.0400,0/2,0.3743


## 9. Zip results (CSVs + plots, no checkpoints) — download from the Output tab

In [9]:
!cd {OUT} && zip -q -r /kaggle/working/lite_results_{EPOCHS}ep.zip . -x '*.keras'
!ls -lh /kaggle/working/lite_results_{EPOCHS}ep.zip

-rw-r--r-- 1 root root 250K Nov 11 20:44 /kaggle/working/lite_results_1500ep.zip
